# **Data Understanding**

## Objectives

Answer business requirement 1:
  * The client is interested in discovering how the house attributes correlate with the sale price. Therefore, the client expects data visualisations of the correlated variables against the sale price to show that.

## Inputs

* outputs/datasets/collection/house_prices 

## Outputs

* generate code that answers BR1 and can be used to build the Streamlit App 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

# Load Data

In [ ]:
import pandas as pd
df = pd.read_csv(f"outputs/datasets/collection/house_prices")
df.head(5)


## Data Exploration

We will generate a Profile Report, to visualise the data and help us examine the data.

In [ ]:
from ydata_profiling import ProfileReport
pandas_report = ProfileReport(df=df, minimal=True)
pandas_report.to_notebook_iframe()

---

## Sales Price - Target Visualisation

It is helpful to explore our target Sales Price, to better understand the nature and distribution of this variable. 

We see that the majority of properties sold for between 100,000 and 200,000. Notably there are outliers in the distribution, leading to right-skewed data.

In [ ]:
import matplotlib.pyplot as plt

df.plot(kind='hist', y='SalePrice', bins=75, figsize=(10,6), grid = 'True', title='Sale Price Distribution')
plt.show()

# Correlation Study

Since we have four categorical variables, we will use Ordinal Encoder to transform them to numerical values. This is a necessary step before running our correlation study, since the method requires variables as numbers.

In ordinal encoding, each unique category value is assigned an integer value. They have a natural ordered relationship, as opposed to One Hot encoding where no ordinal relationship exists.

In [ ]:
from feature_engine.encoding import OrdinalEncoder


encoder = OrdinalEncoder(
    encoding_method='arbitrary',
    variables=['KitchenQual', 'BsmtExposure', 'BsmtFinType1', 'GarageFinish'],
    missing_values='ignore'
)


df_encoded = encoder.fit_transform(df)
print(df_encoded.shape)
df_encoded.head(5)

Due to the skewness of the data, we have opted to use Spearman Correlation over Pearson. Spearman is more appropriate to handle skewed data with extreme outliers, such as our dataset.

In [ ]:
spearman_corr = df_encoded.corr(method='spearman', numeric_only=True)['SalePrice'].sort_values(key=abs, ascending=False)
spearman_corr

## Observations from Correlation Methods

From the above studies, we draw the following observations:
* Two variables, OverallQual and GrLivArea, show a strong correlation with SalesPrice.
* Seven variables (YearBuilt, GarageArea, TotalBsmtSF, GarageYrBlt, 1stFlrSF, YearRemodAdd and OpenPorchSF) show a moderate correlation of 0.48 to 0.65.
* There are a number of variables showing a weak but potentially useful level of correlation for the purposes of Machine Learning.



## Heatmap

We have included a heatmap to help visualise the correlation between the top 10 correlated values and Sales Price.

In [ ]:
# Generate heatmap for spearman correlation
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

top_features = spearman_corr.index[:10]

# Calculate correlation matrix
df_corr = df_encoded[top_features].corr(method='spearman')

# Create mask for upper triangle
mask = np.zeros_like(df_corr, dtype=np.bool_)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask)
plt.title('Top Feature Correlations with Sale Price')
plt.show()



---

## Scatter Plots

We have displayed the data on scatter plots for further visualisation and understanding.

These will help us explore the relationships between each of our top features with Sale Price.

In [ ]:
import seaborn as sns

for feature in top_features[1:]:
    sns.scatterplot(data=df, x=feature, y='SalePrice')
    plt.title(f'{feature} vs Sale Price')
    plt.show()


## Box Plots

To explore the relationship of the categorical variables with Sale Price, we have included 4 box plots.

In [ ]:
categorical_features = ['OverallQual', 'KitchenQual', 'GarageFinish', 'BsmtExposure']

for feature in categorical_features:
    sns.boxplot(data=df, x=feature, y='SalePrice')
    plt.title(f'{feature} vs Sale Price')
    plt.show()


---

# Conclusions

